In [5]:
import os
import pandas as pd
from tqdm import tqdm
from collections import Counter

# Caminho da pasta
DATASETS_ROOT = "/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/multiclass"

def analisar_detalhado_e_resumo():
    files = sorted([f for f in os.listdir(DATASETS_ROOT) if f.endswith('.csv')])
    
    detalhes = []
    lista_contagem = []

    print(f"🔍 Analisando {len(files)} datasets em {DATASETS_ROOT}...\n")

    for file in tqdm(files, desc="Processando"):
        path = os.path.join(DATASETS_ROOT, file)
        try:
            # Lógica rápida: lê apenas a última coluna
            df_sample = pd.read_csv(path, nrows=0)
            target_col_idx = len(df_sample.columns) - 1
            df_target = pd.read_csv(path, usecols=[target_col_idx])
            
            n_classes = df_target.iloc[:, 0].nunique()
            
            detalhes.append({"dataset": file, "n_classes": n_classes})
            lista_contagem.append(n_classes)
            
        except Exception as e:
            print(f"❌ Erro ao processar {file}: {e}")

    # --- 1. PRINT DETALHADO POR DATASET ---
    df_detalhes = pd.DataFrame(detalhes)
    print("\n" + "="*50)
    print("📂 LISTA DETALHADA POR DATASET")
    print("="*50)
    print(df_detalhes.to_string(index=False))

    # --- 2. RELATÓRIO DE FREQUÊNCIA ---
    contagem = Counter(lista_contagem)
    df_resumo = pd.DataFrame(
        list(contagem.items()), 
        columns=['Nº de Classes', 'Qtd de Datasets']
    ).sort_values(by='Nº de Classes')

    print("\n" + "="*50)
    print("📊 RESUMO DE DISTRIBUIÇÃO (FREQUÊNCIA)")
    print("="*50)
    print(df_resumo.to_string(index=False))
    print("="*50)
    print(f"Total de Datasets analisados: {len(detalhes)}")

if __name__ == "__main__":
    analisar_detalhado_e_resumo()

🔍 Analisando 24 datasets em /var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/multiclass...



Processando: 100%|██████████| 24/24 [00:00<00:00, 24.90it/s]


📂 LISTA DETALHADA POR DATASET
             dataset  n_classes
         abalone.csv         11
academic-success.csv          3
           chess.csv         15
             cmc.csv          3
       connect-4.csv          3
          digits.csv         10
        dry-bean.csv          7
     hand_digits.csv         10
             hcv.csv          4
       image_seg.csv          7
          isolet.csv         26
          letter.csv         26
             mhr.csv          3
       molecular.csv          3
         nursery.csv          4
         obesity.csv          7
      page_block.csv          3
        phishing.csv          3
      poker_hand.csv          8
       satellite.csv          6
         shuttle.csv          4
     waveform-v1.csv          3
    wine-quality.csv          5
           yeast.csv          4

📊 RESUMO DE DISTRIBUIÇÃO (FREQUÊNCIA)
 Nº de Classes  Qtd de Datasets
             3                8
             4                4
             5                1
  

In [12]:
import numpy as np
import pickle
import os
import time

# ===========================================================
# =============== MoSS MULTICLASSE (DIRICHLET) ==============
# ===========================================================

def moss_multiclass_dirichlet(n_samples: int, alpha: np.ndarray, merge: float, eps: float = 1e-3):
    n_classes = len(alpha)
    merge = np.clip(merge, 0.0, 1.0)
    scale = 50 * (1 - merge) + 5
    scores = np.zeros((n_samples, n_classes))

    n_per_class = np.floor(n_samples * alpha).astype(int)
    n_per_class[-1] = n_samples - n_per_class[:-1].sum()

    idx = 0
    for c in range(n_classes):
        center = np.full(n_classes, 1.0 / n_classes)
        center[c] = 1.0
        center = center / center.sum()
        conc = eps + scale * ((1 - merge) * center + merge * np.full(n_classes, 1.0 / n_classes))
        samples = np.random.dirichlet(conc, size=n_per_class[c])
        scores[idx:idx + n_per_class[c]] = samples
        idx += n_per_class[c]

    np.random.shuffle(scores)
    return scores

# ===========================================================
# ========== GERADOR DE DISTRIBUIÇÕES MOSS MULTI ============
# ===========================================================

def gerar_distribuicoes_moss_multiclasse(n_samples, n_classes, n_prevalences, n_merges, n_curves, save_path):
    prevalences = np.random.dirichlet(alpha=np.ones(n_classes), size=n_prevalences)
    merges = np.linspace(0.0, 1.0, n_merges)
    synthetic_distributions = {}
    total = len(prevalences) * len(merges)
    count = 0

    print(f"\n🚀 Gerando MoSS para {n_classes} CLASSES -> {save_path}")
    start_global = time.perf_counter()

    for alpha in prevalences:
        alpha_key = tuple(np.round(alpha, 4))
        for merge in merges:
            curves = [moss_multiclass_dirichlet(n_samples, alpha, merge) for _ in range(n_curves)]
            synthetic_distributions[(alpha_key, round(merge, 4))] = curves
            count += 1
            if count % 50 == 0 or count == total:
                print(f"   Progresso: [{count}/{total}] blocos concluídos...")

    with open(save_path, "wb") as f:
        pickle.dump(synthetic_distributions, f, protocol=pickle.HIGHEST_PROTOCOL)

    total_runtime = time.perf_counter() - start_global
    print(f"✔ Finalizado {n_classes} classes em {total_runtime/60:.2f} min\n")

# ===========================================================
# ======================= EXECUÇÃO ==========================
# ===========================================================

if __name__ == "__main__":
    output_dir = "moss_outputs"
    os.makedirs(output_dir, exist_ok=True)

    # Configurações base (iguais para todos)
    base_config = dict(
        n_samples=100,
        n_prevalences=15,
        n_merges=15,
        n_curves=20
    )

    # Loop para gerar os diferentes arquivos solicitados
    classes_para_gerar = [5,6,7,8,10,11,15,26]

    for c in classes_para_gerar:
        file_path = os.path.join(output_dir, f"moss_m_lite_{c}.pkl")
        
        gerar_distribuicoes_moss_multiclasse(
            n_classes=c,
            save_path=file_path,
            **base_config
        )

    print("🏁 Todos os arquivos MoSS multiclasse foram gerados com sucesso.")


🚀 Gerando MoSS para 5 CLASSES -> moss_outputs/moss_m_lite_5.pkl
   Progresso: [50/225] blocos concluídos...
   Progresso: [100/225] blocos concluídos...
   Progresso: [150/225] blocos concluídos...
   Progresso: [200/225] blocos concluídos...
   Progresso: [225/225] blocos concluídos...
✔ Finalizado 5 classes em 0.01 min


🚀 Gerando MoSS para 6 CLASSES -> moss_outputs/moss_m_lite_6.pkl
   Progresso: [50/225] blocos concluídos...
   Progresso: [100/225] blocos concluídos...
   Progresso: [150/225] blocos concluídos...
   Progresso: [200/225] blocos concluídos...
   Progresso: [225/225] blocos concluídos...
✔ Finalizado 6 classes em 0.01 min


🚀 Gerando MoSS para 7 CLASSES -> moss_outputs/moss_m_lite_7.pkl
   Progresso: [50/225] blocos concluídos...
   Progresso: [100/225] blocos concluídos...
   Progresso: [150/225] blocos concluídos...
   Progresso: [200/225] blocos concluídos...
   Progresso: [225/225] blocos concluídos...
✔ Finalizado 7 classes em 0.01 min


🚀 Gerando MoSS para 8 CL

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Carregar os dados
df = pd.read_csv("cross_nclasses_experiment.csv")

# 2. Criar o Boxplot Interativo
fig = px.box(
    df, 
    x="modelo", 
    y="erro", 
    color="modelo",
    points="all",          # Mostra todos os pontos (datasets) ao lado do box
    hover_data=["dataset", "n_classes_original"], # Info extra ao passar o mouse
    title="Distribuição do Erro de Quantificação (MAE) por Modelo",
    labels={"erro": "Erro Médio Absoluto (MAE)", "modelo": "Algoritmo/Configuração"},
    category_orders={"modelo": ["EMQ", "MoSS_3", "MoSS_4", "MoSS_7"]} # Organiza a ordem
)

# 3. Customizar o layout para parecer um paper científico
fig.update_layout(
    template="plotly_white",
    showlegend=False,
    xaxis_title="Modelo",
    yaxis_title="Erro (MAE)",
    font=dict(family="Arial", size=14)
)

# 4. Exibir o gráfico
fig.show()

# 5. Opcional: Salvar como HTML (interativo) para abrir no navegador
pio.write_html(fig, file='1.html', auto_open=True)

In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Carregar os dados
df = pd.read_csv("cross_nclasses_experiment.csv")

# 2. Configurações de layout
datasets = df['dataset'].unique()
n_datasets = len(datasets)
# Criamos uma subfigura por dataset, em uma única coluna
fig = make_subplots(
    rows=n_datasets, cols=1, 
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02 # Espaço curto entre os gráficos
)

# 3. Iterar e adicionar cada gráfico com sua própria ordem
for i, ds in enumerate(datasets, 1):
    df_ds = df[df['dataset'] == ds].copy()
    
    # Calcular a ordem local (pela mediana do erro neste dataset)
    ordem_local = df_ds.groupby("modelo")["erro"].median().sort_values().index.tolist()
    
    # Adicionar um boxplot para cada modelo, seguindo a ordem local
    for modelo in ordem_local:
        df_mod = df_ds[df_ds['modelo'] == modelo]
        fig.add_trace(
            go.Box(
                y=df_mod['erro'],
                name=modelo,
                boxpoints='outliers',
                legendgroup=modelo,
                showlegend=(i == 1) # Só mostra a legenda no primeiro gráfico
            ),
            row=i, col=1
        )

# 4. Ajustes finais de tamanho e estética
fig.update_layout(
    height=n_datasets * 300, # 300px para cada dataset
    template="plotly_white",
    title_text="Performance Local: Modelos Ordenados do Melhor para o Pior por Dataset",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

# Deixar os eixos X independentes para cada subgráfico respeitar sua ordem
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="MAE")
fig.write_html("meu_resultado_ordenado.html")
fig.show()
pio.write_html(fig, file='2.html', auto_open=True)

In [11]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

# 1. Carregar e Limpar os dados
df = pd.read_csv("cross_nclasses_experiment.csv")
df['erro'] = pd.to_numeric(df['erro'], errors='coerce')
df = df.dropna(subset=['erro', 'modelo', 'dataset'])

# 2. Configurações de layout
datasets = df['dataset'].unique()
n_datasets = len(datasets)
modelos = df['modelo'].unique()
n_repeats = 30

# Cores fixas
cores = px.colors.qualitative.Dark24
color_map = {modelo: cores[i % len(cores)] for i, modelo in enumerate(modelos)}

# --- AJUSTE DO ESPAÇAMENTO BRANCO ---
# Quanto maior a altura, menor deve ser o vertical_spacing relativo.
# 20 pixels de espaço real dividido pela altura total:
espaco_em_pixels = 100 
altura_por_dataset = 600 # 1600 era muito, mas se quiser manter, o cálculo abaixo ajusta
altura_total = n_datasets * altura_por_dataset
spacing_calculado = espaco_em_pixels / altura_total 

fig = make_subplots(
    rows=n_datasets, cols=1, 
    subplot_titles=[f"<b>Dataset: {ds}</b>" for ds in datasets],
    vertical_spacing=spacing_calculado # Espaçamento fixo em pixels convertido para relativo
)

# 3. Iterar por dataset e construir os boxplots
for i, ds in enumerate(datasets, 1):
    df_ds = df[df['dataset'] == ds].copy()
    df_ds['prevalencia_id'] = df_ds.groupby('modelo').cumcount() // n_repeats + 1
    
    for modelo in modelos:
        df_plot = df_ds[df_ds['modelo'] == modelo]
        
        if not df_plot.empty:
            fig.add_trace(
                go.Box(
                    x=df_plot['prevalencia_id'],
                    y=df_plot['erro'],
                    name=modelo,
                    marker_color=color_map[modelo],
                    boxpoints='outliers',
                    legendgroup=modelo,
                    showlegend=(i == 1),
                    offsetgroup=modelo
                ),
                row=i, col=1
            )

# 4. Ajustes Finais
fig.update_layout(
    height=altura_total,
    width=1300,
    template="plotly_white",
    title_text="Análise de Erro por Prevalência (Detalhamento por Dataset)",
    boxmode='group',
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.005, xanchor="right", x=1),
    # Diminuir as margens internas de cada subplot para usar mais espaço
    margin=dict(t=300, b=100, l=60, r=40)
)

# Ajustar títulos das subfiguras (colocar mais perto do gráfico)
for i in range(len(fig.layout.annotations)):
    fig.layout.annotations[i].update(yshift=-10) # Empurra o título para baixo, mais perto do box

# Garantir eixos X visíveis
fig.update_xaxes(tickmode='linear', dtick=1, title_text="ID da Prevalência")
fig.update_yaxes(title_text="MAE")

# 5. Salvar
fig.write_html("resultado_sem_espacos.html")
print(f"✅ HTML gerado com spacing de {spacing_calculado:.5f}")
fig.show()

✅ HTML gerado com spacing de 0.00694


In [22]:
import pandas as pd
import plotly.express as px
import numpy as np

# ============================================================
# 1. CONFIGURAÇÕES (Devem ser iguais às do seu experimento)
# ============================================================
REPEATS = 30  # Número de repetições por prevalência (protocolo.repeats)
DATASET_ALVO = 'molecular.csv'
MODELOS_ALVO = ['EMQ', 'MoSS_3']

# ============================================================
# 2. CARREGAR E FILTRAR
# ============================================================
df = pd.read_csv("cross_nclasses_experiment.csv")

# Filtrar apenas o dataset molecular e os modelos EMQ e MoSS_3
df_filtered = df[(df['dataset'] == DATASET_ALVO) & 
                 (df['modelo'].isin(MODELOS_ALVO))].copy()

# ============================================================
# 3. ORGANIZAR IDs DE PREVALÊNCIA
# ============================================================
# Ordenamos para garantir que o contador 'cumcount' siga a sequência de execução
df_filtered = df_filtered.sort_values(['modelo', 'dataset'])

# Criar o ID (1, 2, 3...) baseado no número de repetições
# O UPP processa 'N' prevalências, cada uma repetida 'REPEATS' vezes.
df_filtered['prevalencia_id'] = df_filtered.groupby('modelo').cumcount() // REPEATS + 1

# ============================================================
# 4. GERAR O GRÁFICO (Visual Acadêmico)
# ============================================================
fig = px.box(
    df_filtered, 
    x="prevalencia_id", 
    y="erro", 
    color="modelo",
    title=f"Comparação Detalhada: {DATASET_ALVO} (EMQ vs MoSS_3)",
    labels={
        "prevalencia_id": "ID da Prevalência (Lote do Protocolo UPP)",
        "erro": "Erro Médio Absoluto (MAE)",
        "modelo": "Algoritmo"
    },
    boxmode="group",      # Coloca os modelos lado a lado em cada ID
    points="outliers"     # "outliers" para visual limpo, "all" para depuração
)

# ============================================================
# 5. AJUSTES DE LAYOUT E SALVAMENTO
# ============================================================
fig.update_layout(
    template="plotly_white",
    height=600,
    width=1200,
    xaxis=dict(
        tickmode='linear', 
        dtick=1,
        title="Distribuição de Prevalência (ID único do Simplex)"
    ),
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=1.02, 
        xanchor="right", 
        x=1
    )
)

# Salvar e mostrar
fig.write_html("resultado_final_molecular.html")
fig.show()

print(f"✅ Gráfico gerado com sucesso para {DATASET_ALVO}!")
print(f"📊 Total de pontos por modelo: {len(df_filtered)/2}")

✅ Gráfico gerado com sucesso para molecular.csv!
📊 Total de pontos por modelo: 300.0
